### Experiment with o4-mini

In [ ]:
import pandas as pd
import libs.prompts as prompts
from libs.exp_utils import classify_dataset, evaluate_experiment
from libs.openai import call_openai
from tqdm import tqdm
import json
import os

In [ ]:
# Define model name
MODEL_NAME = "o4-mini"  # Change to the desired model name

### Dataset load

In [ ]:
# Load ground truth bug locations
with open("../../Datasets/bugLocationDappScan.json", "r") as f:
    bug_locations = json.load(f)

In [ ]:
def normalize_path(path):
    if isinstance(path, str):
        return path.replace('\\', '/').replace('./', '').replace('.\\', '')
    return path

bugloc_dict = {
    normalize_path(item['file']): normalize_path(item['location'])
    for item in bug_locations
}


# Load the dataset and convert to appropriate data types
csv_path = "../../Datasets/dw.csv"
df = pd.read_csv(csv_path)
df['has_error'] = df['has_error'].astype(bool)
subset = df.reset_index(drop=True)
#subset.head()

#### Zero-shot Prompting

In [ ]:
# Step 1: Binary error detection (como antes)
results_df = classify_dataset(subset, prompts.create_zeroshot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

In [ ]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

In [ ]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path]) #.replace('.sol', '.json'))
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [ ]:
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    #llm_categories.append(row['location'])  # Assuming 'bug_type' is the LLM's prediction
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

   # print("LOCATION: LINE - ", row['location'])
#print("LLM Categories:", llm_categories, "GT Categories:", gt_categories)


In [ ]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")


#### Zero-shot CoT Prompting

In [ ]:
results_df = classify_dataset(subset, prompts.create_zeroshot_cot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

In [ ]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

In [ ]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [ ]:
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    #llm_categories.append(row['location'])  # Assuming 'bug_type' is the LLM's prediction
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

   # print("LOCATION: LINE - ", row['location'])
#print("LLM Categories:", llm_categories, "GT Categories:", gt_categories)


In [ ]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

#### Zero-shot ToT Prompting

In [ ]:
results_df = classify_dataset(subset, prompts.create_zeroshot_tot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

In [ ]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

In [ ]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [ ]:
# Roda a checagem
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    #llm_categories.append(row['location'])  # Assuming 'bug_type' is the LLM's prediction
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

   # print("LOCATION: LINE - ", row['location'])
#print("LLM Categories:", llm_categories, "GT Categories:", gt_categories)


In [ ]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")